# Part 1: Extending the context window of EuroMoE-2.6B-A0.6B

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return None

for name in ["WANDB_API_KEY", "HF_TOKEN"]:
    if value := secret(name):
        os.environ[name] = value

REPO_DIR = "/kaggle/working/ai500-labs"
BRANCH = "main"
url = "https://github.com/PavloKhomenko/ai500-labs.git"

In [2]:
# the token is passed only on the command line, origin keeps the public url (/kaggle/working gets saved)
if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q {url} {BRANCH}
else:
    !git clone -q -b {BRANCH} {url} {REPO_DIR}
    !git -C {REPO_DIR} remote set-url origin https://github.com/PavloKhomenko/ai500-labs.git
!git -C {REPO_DIR} log --oneline -1

94434b3 (HEAD -> main, origin/main, origin/HEAD) quick fix


In [3]:
# versions from lab-1/uv.lock; torch stays Kaggle's CUDA build
!pip install -q "transformers==5.18.0" "datasets==5.1.0" "accelerate==1.15.0" "peft==0.21.2" \
    "huggingface-hub==1.33.0" "wandb==0.30.0" "tabulate==0.10.0" "pyyaml==6.0.3"
# Kaggle ships torchao 0.10; peft 0.21 needs torchao > 0.16 and errors while dispatching LoRA layers, and we don't use it
!pip uninstall -q -y torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 38.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not c

In [4]:
import sys
sys.path.insert(0, f"{REPO_DIR}/lab-1/src")

import pandas as pd
import torch
import transformers
from transformers import AutoConfig, AutoTokenizer

from lab1.common import load_config, out_dir
from lab1.data import prepare_part1_data
from lab1.part2 import wandb_log as wl
from lab1.part2.pretrain import load_model
from lab1.part1.rope import RoPEScaling
from lab1.part1.evaluate import LongContextEvaluator, ResultsReport
from lab1.part1.kv_cache import KVCacheProbe
from lab1.part1.finetune import LongLoRATrainer
%matplotlib inline

cfg = load_config()
p1 = cfg["part1"]
device = torch.device("cuda")
env = {"torch": torch.__version__, "transformers": transformers.__version__, "gpu": torch.cuda.get_device_name()}
print(env)

def start_run(name, job_type, **extra):
    return wl.init_run(cfg, name, job_type, config={**p1, **env, **extra}, group="part1",
                       project=cfg["wandb"]["part1_project"])

{'torch': '2.11.0+cu128', 'transformers': '5.18.0', 'gpu': 'Tesla T4'}


## 0–1. RoPE

In [5]:
config = AutoConfig.from_pretrained(p1["model"])
rope = RoPEScaling.from_config(config, p1["scale"])
rope.describe()
rope.compare_with_hf(config)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

base=500000 rotary_dim=128 L_train=4096 s=4
pairs with lambda_i > L_train: 32 / 64 (first i=32), lambda_i > s*L_train: 25
none         scaling=1.0000 inv_freq[0,16,32,48,63]=['1.000e+00', '3.761e-02', '1.414e-03', '5.318e-05', '2.455e-06']
linear       scaling=1.0000 inv_freq[0,16,32,48,63]=['2.500e-01', '9.402e-03', '3.536e-04', '1.330e-05', '6.138e-07']
ntk          scaling=1.0000 inv_freq[0,16,32,48,63]=['1.000e+00', '2.645e-02', '6.994e-04', '1.850e-05', '6.138e-07']
dynamic_ntk  scaling=1.0000 inv_freq[0,16,32,48,63]=['1.000e+00', '1.960e-02', '3.843e-04', '7.534e-06', '1.889e-07']
yarn         scaling=1.1386 inv_freq[0,16,32,48,63]=['1.000e+00', '3.447e-02', '3.536e-04', '1.330e-05', '6.138e-07']


,method,seq_len,max_abs_diff,scaling_ours,scaling_hf
0,linear,NaN,0.0,1.000000,1.000000
1,yarn,NaN,0.0,1.138629,1.138629
2,dynamic_ntk,4096.0,0.0,1.000000,1.000000
3,dynamic_ntk,8192.0,0.0,1.000000,1.000000
4,dynamic_ntk,16384.0,0.0,1.000000,1.000000
5,ntk,NaN,0.0,NaN,NaN


In [6]:
tok = AutoTokenizer.from_pretrained(p1["model"])
data = prepare_part1_data(cfg, tok, out_dir(cfg, "part1", "data"))
{k: tuple(v.shape) for k, v in data.items() if torch.is_tensor(v)}

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 15.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

tokenizer.model: reconstructing file:   0%|          |  0.00B / 2.41MB            

tokenizer.model: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-29a571947c0b5cc(…): reconstructing file:   0%|          |  0.00B / 24.9MB            

data/test-00000-of-00001-29a571947c0b5cc(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

94 documents with >= 16384 tokens
part1 data: uk (100, 16385), en (94, 16385), train (800, 8192)


{'uk': (100, 16385),
 'en': (94, 16385),
 'train': (800, 8192),
 'uk_bytes': (100, 3),
 'en_bytes': (94, 3)}

In [7]:
model = load_model(p1["model"], device)
evaluator = LongContextEvaluator(model, tok, p1, seed=cfg["seed"])
eval_dir, fig_dir = out_dir(cfg, "part1", "eval"), out_dir(cfg, "part1", "figures")

for method in p1["methods"]:
    start_run(f"eval-{method}", "eval", method=method)
    evaluator.evaluate(method, data, eval_dir)
    wl.finish()

model.safetensors.index.json:   0%|          | 0.00/459k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

wandb: uploading artifact run-clyzcyb1-passkey; updating run metadata
wandb: uploading artifact run-clyzcyb1-passkey
wandb: 
wandb: Run history:
wandb:  loss/en ▁▁▁▁▁▁▁▁▂▃▇▇▇███
wandb:  loss/uk ▁▁▁▁▁▁▁▁▂▃▇▇████
wandb: position ▁▁▂▂▃▃▄▄▅▅▆▆▇▇██▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
wandb: 
wandb: Run summary:
wandb:    bpb/en_16384 1.51952
wandb:     bpb/en_4096 0.88893
wandb:     bpb/en_8192 0.8911
wandb:    bpb/uk_16384 0.92511
wandb:     bpb/uk_4096 0.47091
wandb:     bpb/uk_8192 0.46965
wandb:         loss/en 6.54503
wandb:         loss/uk 6.40441
wandb: minutes/loss_en 4.76393
wandb: minutes/loss_uk 5.05279
wandb:             +20 ...
wandb: 
wandb: 🚀 View run eval-ntk at: https://wandb.ai/ai340-nlp/ai500-lab1-part1/runs/clyzcyb1
wandb: ⭐️ View project at: https://wandb.ai/ai340-nlp/ai500-lab1-part1
wandb: Synced 5 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261013_145131-clyzcyb1/logs
wandb: setting up run qprnjrn2
wandb: Tracking run with wand

[dynamic_ntk uk] first document 4.8s -> ~7.9 min for 100
[dynamic_ntk en] first document 4.8s -> ~7.5 min for 94
rope=dynamic_ntk s=4 seq_len=4096: inv_freq[:3]=[1.0, 0.8146172165870667, 0.663601279258728] inv_freq[-3:]=[3.6997230381530244e-06, 3.01385807688348e-06, 2.4551407022954663e-06] attention_scaling=1.0000
passkey dynamic_ntk uk L=4096 depth=0.1: 20/20
passkey dynamic_ntk uk L=4096 depth=0.5: 20/20
passkey dynamic_ntk uk L=4096 depth=0.9: 20/20
rope=dynamic_ntk s=4 seq_len=8192: inv_freq[:3]=[1.0, 0.7940700650215149, 0.630547285079956] inv_freq[-3:]=[7.787333515807404e-07, 6.183688014971267e-07, 4.910281745651446e-07] attention_scaling=1.0000
passkey dynamic_ntk uk L=8192 depth=0.1: 19/20
passkey dynamic_ntk uk L=8192 depth=0.5: 20/20
passkey dynamic_ntk uk L=8192 depth=0.9: 20/20
rope=dynamic_ntk s=4 seq_len=16384: inv_freq[:3]=[1.0, 0.7821174263954163, 0.6117076277732849] inv_freq[-3:]=[3.087373272592231e-07, 2.414688538010523e-07, 1.888569869379353e-07] attention_scaling=1.0

wandb: uploading artifact run-qprnjrn2-passkey; updating run metadata
wandb: uploading artifact run-qprnjrn2-passkey
wandb: uploading media/table/passkey_32_0916dcab5f2cb893192d.table.json; uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:  loss/en ▁▂▃▂▃▃▃▃▄▄▄▄▅▅▇█
wandb:  loss/uk ▂▁▁▁▁▁▁▁▂▂▂▂▄▆▇█
wandb: position ▁▁▂▂▃▃▄▄▅▅▆▆▇▇██▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
wandb: 
wandb: Run summary:
wandb:    bpb/en_16384 0.96621
wandb:     bpb/en_4096 0.87431
wandb:     bpb/en_8192 0.89363
wandb:    bpb/uk_16384 0.50575
wandb:     bpb/uk_4096 0.467
wandb:     bpb/uk_8192 0.46614
wandb:         loss/en 2.92505
wandb:         loss/uk 2.32047
wandb: minutes/loss_en 7.45398
wandb: minutes/loss_uk 7.93296
wandb:             +20 ...
wandb: 
wandb: 🚀 View run eval-dynamic_ntk at: https://wandb.ai/ai340-nlp/ai500-lab1-part1/runs/qprnjrn2
wandb: ⭐️ View project at: https://wandb.ai/ai340-nlp/ai500-lab1-part1
wandb: Synced 5 W&B file(s), 1 media file(s), 2 artifa

rope=yarn s=4 seq_len=None: inv_freq[:3]=[1.0, 0.8146172165870667, 0.663601279258728] inv_freq[-3:]=[9.249307595382561e-07, 7.5346451922087e-07, 6.137851755738666e-07] attention_scaling=1.1386
[yarn uk] first document 3.1s -> ~5.1 min for 100
rope=yarn s=4 seq_len=None: inv_freq[:3]=[1.0, 0.8146172165870667, 0.663601279258728] inv_freq[-3:]=[9.249307595382561e-07, 7.5346451922087e-07, 6.137851755738666e-07] attention_scaling=1.1386
[yarn en] first document 3.0s -> ~4.7 min for 94
rope=yarn s=4 seq_len=None: inv_freq[:3]=[1.0, 0.8146172165870667, 0.663601279258728] inv_freq[-3:]=[9.249307595382561e-07, 7.5346451922087e-07, 6.137851755738666e-07] attention_scaling=1.1386
passkey yarn uk L=4096 depth=0.1: 20/20
passkey yarn uk L=4096 depth=0.5: 20/20
passkey yarn uk L=4096 depth=0.9: 20/20
passkey yarn uk L=8192 depth=0.1: 20/20
passkey yarn uk L=8192 depth=0.5: 20/20
passkey yarn uk L=8192 depth=0.9: 20/20
passkey yarn uk L=16384 depth=0.1: 20/20
passkey yarn uk L=16384 depth=0.5: 20/20


wandb: uploading artifact run-2a0safwm-passkey; updating run metadata
wandb: uploading artifact run-2a0safwm-passkey
wandb: 
wandb: Run history:
wandb:  loss/en ▁▄▅▅▅▅▅▆▅▅▆▆▆▆▆█
wandb:  loss/uk █▁▃▃▂▂▃▂▃▁▄▅▇▅▄▅
wandb: position ▁▁▂▂▃▃▄▄▅▅▆▆▇▇██▁▁▂▂▃▃▄▄▅▅▆▆▇▇██
wandb: 
wandb: Run summary:
wandb:    bpb/en_16384 0.91205
wandb:     bpb/en_4096 0.89715
wandb:     bpb/en_8192 0.89926
wandb:    bpb/uk_16384 0.4784
wandb:     bpb/uk_4096 0.47695
wandb:     bpb/uk_8192 0.47175
wandb:         loss/en 2.49181
wandb:         loss/uk 1.84233
wandb: minutes/loss_en 4.77351
wandb: minutes/loss_uk 5.08155
wandb:             +20 ...
wandb: 
wandb: 🚀 View run eval-yarn at: https://wandb.ai/ai340-nlp/ai500-lab1-part1/runs/2a0safwm
wandb: ⭐️ View project at: https://wandb.ai/ai340-nlp/ai500-lab1-part1
wandb: Synced 5 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261013_153646-2a0safwm/logs


In [8]:
start_run("kv-cache", "analysis")
kv = KVCacheProbe(model).table([p1["l_train"] * k for k in (1, 2, 4)], out_dir(cfg, "part1", "kv_cache") / "kv_cache")
wl.finish()
kv

wandb: setting up run s65p7ub7
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261013_155639-s65p7ub7
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run kv-cache
wandb: ⭐️ View project at https://wandb.ai/ai340-nlp/ai500-lab1-part1
wandb: 🚀 View run at https://wandb.ai/ai340-nlp/ai500-lab1-part1/runs/s65p7ub7
wandb: updating run metadata; uploading artifact run-s65p7ub7-kv_cache
wandb: uploading artifact run-s65p7ub7-kv_cache
wandb: 🚀 View run kv-cache at: https://wandb.ai/ai340-nlp/ai500-lab1-part1/runs/s65p7ub7
wandb: ⭐️ View project at: https://wandb.ai/ai340-nlp/ai500-lab1-part1
wandb: Synced 4 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20261013_155639-s65p7ub7/logs


,seq_len,predicted_MiB,cache_tensors_MiB,allocated_increase_MiB,peak_increase_MiB,allocated_over_predicted
0,4096,96.0,96.0,96.0,515.907227,1.0
1,8192,192.0,192.0,192.0,1031.813477,1.0
2,16384,384.0,384.0,384.0,2063.625977,1.0


In [9]:
del model, evaluator
torch.cuda.empty_cache()
ft_dir = out_dir(cfg, "part1", "finetune")

start_run(f"finetune-{p1['finetune']['method']}", "train", **p1["finetune"])
trainer = LongLoRATrainer(load_model(p1["model"], device), p1, seed=cfg["seed"])
trainer.train(data["train"], ft_dir / "train_log.jsonl", max_steps=None)  # max_steps=5 for timing
trainer.save(ft_dir / "trainable.pt")
wl.finish()

wandb: setting up run c39azet2
wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20261013_155646-c39azet2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run finetune-yarn
wandb: ⭐️ View project at https://wandb.ai/ai340-nlp/ai500-lab1-part1
wandb: 🚀 View run at https://wandb.ai/ai340-nlp/ai500-lab1-part1/runs/c39azet2


Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

rope=yarn s=4 seq_len=None: inv_freq[:3]=[1.0, 0.8146172165870667, 0.663601279258728] inv_freq[-3:]=[9.249307595382561e-07, 7.5346451922087e-07, 6.137851755738666e-07] attention_scaling=1.1386


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
del trainer
torch.cuda.empty_cache()
method = p1["finetune"]["method"]
model = LongLoRATrainer.load_finetuned(load_model(p1["model"], device), ft_dir / "trainable.pt", p1)
evaluator = LongContextEvaluator(model, tok, p1, seed=cfg["seed"])

start_run(f"eval-{method}-ft", "eval", method=method, finetuned=True)
evaluator.evaluate(method, data, eval_dir, tag=f"{method}-ft")
wl.finish()

In [ ]:
report = ResultsReport(eval_dir, fig_dir, p1["l_train"], p1["bin_size"])
start_run("eval-summary", "analysis")
report.loss_plot("uk")
report.loss_plot("en")
bpb, passkey = report.bpb_table(), report.passkey_table()
wl.finish()
display(bpb, passkey)